In [56]:
import pandas as pd

In [57]:
df = pd.read_csv("../data/SMSSpamCollection", sep="\t", header=None,
                 names=["label", "text"], quoting=3)
df.shape

(5574, 2)

In [58]:
df["label"].value_counts(normalize=True)

label
ham     0.865985
spam    0.134015
Name: proportion, dtype: float64

In [59]:
df_clean = df.drop_duplicates().copy()
df_clean["label_num"] = df_clean["label"].map({"ham": 0, "spam": 1})

In [60]:
df_clean.shape

(5171, 3)

In [61]:
conflicts = df.groupby("text")["label"].nunique () > 1

In [62]:
conflicts.sum()


np.int64(0)

In [63]:
conflicts[conflicts].index

Index([], dtype='object', name='text')

In [64]:
from sklearn.model_selection import train_test_split

X = df_clean["text"]
y = df_clean["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
df_clean["label_num"].value_counts()

label_num
0    4518
1     653
Name: count, dtype: int64

In [67]:
df_clean["label_num"].isna().sum()

np.int64(0)

In [68]:
print(X_train.shape, X_test.shape)
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)
len(set(X_train) & set(X_test))


(4136,) (1035,)


0

In [73]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))
print(len(set(X_train) & set(X_test))) 

(4136,) (1035,)
label_num
0    0.873791
1    0.126209
Name: proportion, dtype: float64
label_num
0    0.87343
1    0.12657
Name: proportion, dtype: float64
0


# Knowledge Check 2: Clean and split 

## What I did
1. Reloaded the data with `quoting=3` (5,574 rows)
2. Dropped exact duplicates: 5,574 to 5,171 rows (403 removed)
3. Encoded the label: ham = 0, spam = 1 and named it (`label_num`)
4. Split 80/20 with `stratify` and `random_state=42`: 4,136 train / 1,035 test

## Results
- After dropping duplicates: 4,518 ham, 653 spam (~12.6% spam)
- Train is 12.62% spam, test is 12.66% spam, so both match the overall number
- Overlap between train and test messages: **0**
- Test set has about 131 scams. Each missed scam moves recall by under 1 point, so recall is usable, but a 1-2 point difference between models could just be noise.

## Decisions
- **Dropped duplicates** because the same message in train and test would let the model remember instead of learn (data leakage). Spam repeated about 2x as often as ham (12.6% vs 6.4%) since spammers send the same template to everyone.
- **Checked label conflicts** on the full data: no text appears with both labels, so dropping duplicates didn't hide any labeling problems.
- **Spam = 1** because it's the positive class (the thing I'm trying to catch), and recall metric treats 1 as positive.
- **Not cleaning the text yet** (like lowercasing). Spam uses caps like "FREE" and "WINNER", so I'll decide in Step 3 using real results instead of guessing.

## Key terms
- **label_num:** the label as a number so the model can use it. I made it with `.map()` and kept the original `label` column.
- **Positive class:** what I'm trying to catch (spam = 1).
- **X vs y:** X is the input (message text), y is the target (label_num).
- **Train/test split:** the model learns from train and I check it on test messages it has never seen.
- **test_size:** the fraction that goes to the test set (0.2 = 20%).
- **random_state:** a seed for the shuffle. Same number = same split every time, so results are reproducible.
- **stratify:** keeps the spam percentage the same in train and test. Without it, a random split could leave too few scams in the test set and make recall unreliable.
- **Data leakage:** when the model gets info it shouldn't have (like the same message in train and test), so the test score looks better than it really is.
- **SettingWithCopyWarning:** pandas isn't sure my DataFrame is its own copy. Fix: `.copy()` after dropping or filtering.


## Limitations
- I only dropped **exact** duplicates. Messages that differ by case or spacing ("Ok" vs "ok ") could still land on opposite sides of the split and leak a little. I can measure this later.
- The dataset is old UK/Singapore SMS, so it may not catch modern scam wording.